# **Part B: Multi-Period Approach Using MSP (Multi-Stage Stochastic Programming)**

---

### 1. Problem Formulation: Two-Stage MSP for Retirement Planning

**Assumptions:**

* Investor age: 45
* Planning horizon: 20 years, with time steps of 10 years
* Strategy: Buy-and-hold within each 10-year stage
* Objective: Maximize expected utility of final wealth

* Income grows at 3% annually (above inflation).
* Investor contributes a fixed proportion of salary each year.

---
#### 1. Sets

* \$T = {1, 2}\$: Time stages (0-10, 10-20 years)
  * Stage 1 (t=0\~10 years): Current age 45 → age 55
  * Stage 2 (t=10\~20 years): Age 55 → retirement at 65
* \$A = {\text{Stocks}, \text{Bonds}, \text{Commodities}, \text{Cash}}\$: Asset classes
* \$\Omega\$: Set of scenarios (simulated market returns)

#### 2. Decision Variables

* \$x\_{a}^{(1)}\$: Initial allocation to asset \$a\$ at \$t=0\$
* \$x\_{a}^{(2, \omega)}\$: Reallocation to asset \$a\$ at \$t=10\$ in scenario \$\omega\$
* \$w^{(2, \omega)}\$: Final wealth in scenario \$\omega\$ at \$t=20\$

#### 3. Objective Function

Maximize expected final wealth:

$$
\max \sum_{\omega \in \Omega} p_\omega \cdot w^{(2, \omega)}
$$

#### 4. Constraints

1. **Stage 1 budget constraint**:

$$
\sum_{a \in A} x_{a}^{(1)} = W_0
$$

2. **Stage 2 wealth definition** (for each scenario \$\omega\$):

$$
\sum_{a \in A} x_{a}^{(2, \omega)} = \sum_{a \in A} x_{a}^{(1)} \cdot (1 + r_{a}^{(1, \omega)}) + \sum_{t=11}^{20} 0.12 \cdot s_t
$$

3. **Final wealth calculation**:

$$
w^{(2, \omega)} = \sum_{a \in A} x_{a}^{(2, \omega)} \cdot (1 + r_{a}^{(2, \omega)})
$$

4. **Non-negativity**:

$$
x_{a}^{(1)} \geq 0, \quad x_{a}^{(2, \omega)} \geq 0 \quad \forall a, \omega
$$


---

### 2. Numerical Solution and Interpretation (Python with Monte Carlo + Optimization)

**Implementation overview:**

* 1000 random return paths for each asset, per decade (based on historical mean & covariance)
* Linear programming used to find optimal $x_{a}^{(1)}$ and recourse $x_{a}^{(2, \omega)}$
* Output: Distribution of final wealth across scenarios


In [4]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# 시뮬레이션 설정
np.random.seed(42)
n_assets = 4
n_scenarios = 100

# 투자 시작 직전인 5년치 정보 이용 ("Assume annual time steps (starting January 1, 2013)" 조건 반영)
# - Part A에서 구해놓은 연간수익률, 공분산 사용
mean_returns_stage1 = np.array([0.05, 0.06, -0.03, 0.01])
mean_returns_stage2 = np.array([0.05, 0.06, -0.03, 0.01])
cov_matrix = np.diag([0.058, 0.0036, 0.0759, 0.000001])   # 상관관계 0이라 가정

# 자산 클래스: Stocks, Bonds, Commodities, Cash
initial_salary = 160000
growth_rate = 0.03
saving_rate = 0.12
years_1 = np.arange(1, 11)
years_2 = np.arange(11, 21)

# Stage 1 누적 저축
salaries_1 = initial_salary * (1 + growth_rate) ** (years_1 - 1)
contrib_1 = np.sum(saving_rate * salaries_1)
initial_wealth = 550000 + contrib_1

# 시나리오별 수익률 생성
stage1_returns = np.random.multivariate_normal(mean_returns_stage1, cov_matrix, n_scenarios)
stage2_returns = np.random.multivariate_normal(mean_returns_stage2, cov_matrix, n_scenarios)

# Stage 2 누적 저축 (같은 시나리오별로 동일하게 적용됨)
salaries_2 = initial_salary * (1 + growth_rate) ** (years_2 - 1)
contrib_2 = np.sum(saving_rate * salaries_2)

# 목표 자산
target_wealth = 4000000

# linprog를 사용한 2단계 최적화 (시나리오별)
results = []
for i in range(n_scenarios):
    r1 = stage1_returns[i]
    r2 = stage2_returns[i]

    # 목적함수: maximize final wealth -> minimize -w2
    c = np.zeros(2 * n_assets)
    c[n_assets:] = - (1 + r2)  # stage 2 weights

    # Equality constraint 1: stage 1 budget
    A_eq = [np.concatenate([np.ones(n_assets), np.zeros(n_assets)])]
    b_eq = [initial_wealth]

    # Equality constraint 2: stage 2 budget (from stage1 growth + contrib2)
    wealth_after_stage1 = (1 + r1) * initial_wealth
    rhs_stage2 = np.sum(wealth_after_stage1) + contrib_2
    A_eq.append(np.concatenate([np.zeros(n_assets), np.ones(n_assets)]))
    b_eq.append(rhs_stage2)

    # Bounds: no short selling
    bounds = [(0, None)] * (2 * n_assets)

    res = linprog(c=c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

    if res.success:
        w2 = -res.fun
        results.append(w2)

# 결과 해석
results = np.array(results)
expected_final_wealth = np.mean(results)
std_final_wealth = np.std(results)
prob_success = np.mean(results >= target_wealth)

# 요약
summary = pd.DataFrame({
    "Measure": ["Expected Final Wealth", "Standard Deviation", "Probability of Success"],
    "Value": [expected_final_wealth, std_final_wealth, prob_success]
})

summary


,Measure,Value
0,Expected Final Wealth,4.103006e+06
1,Standard Deviation,5.690528e+05
2,Probability of Success,5.200000e-01


####  Interpret the outcomes

| Measure                    | Value          | What It Means in Simple Terms                                                                                                                      |
| -------------------------- | -------------- | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Expected Final Wealth**  | \$4.10 million | On average, this investor is expected to have about **\$4.1 million** in savings at retirement, based on how investments might perform.            |
| **Standard Deviation**     | \$569,053      | This number shows how much the final wealth can **vary** due to market uncertainty. In most cases, the result will fall between \$3.5M and \$4.7M. |
| **Probability of Success** | **52%**        | There is a **52% chance** that the investor’s wealth will reach or exceed the **target of \$4 million** needed for a comfortable retirement.       |


#### What Does This Mean for the Investor?

Imagine running the retirement plan 100 times in different possible future market conditions:

- In about half of the cases, the investor reaches their retirement goal.
- In the other half, they fall short, possibly needing to adjust spending or save more.
- The "expected" outcome is just an average, not a guarantee.

"The plan now isn't bad, but it's not safe. It's half-and-half odds, so if you want to retire more reliably, you'd better consider a strategy of increasing savings or working longer!"
